# Этап 3, версия 1.2. Проверка роли площади

Проверяем гипотезу на квартирах: помогает ли заполнение отсутствующей площади или лучше обучаться только на объектах с известной площадью? Сравниваем одинаковый алгоритм LightGBM на одной и той же validation. Финальный test не используется.

## Три варианта обучения

A — все квартиры, площадь заполнена по правилам этапа 2, флаг пропуска сохранён. B — только квартиры с исходно известной площадью. C — все квартиры, но площадь и её флаг исключены из признаков. Варианты A и B оцениваются на одних и тех же квартирах с известной площадью; A и C дополнительно сравниваются там, где площадь отсутствует.

In [1]:
# Загружаем подготовку этапа 2 v1.5 без повторного показа его графиков.
import json
from pathlib import Path
from IPython.utils.capture import capture_output
stage2=next((p/"notebooks"/"02_preprocessing_v1.5.ipynb"
             for p in (Path.cwd(),Path.cwd().parent)
             if (p/"notebooks"/"02_preprocessing_v1.5.ipynb").exists()),None)
if stage2 is None:
    raise FileNotFoundError("Не найден этап 2 v1.5")
with capture_output():
    for cell in json.loads(stage2.read_text(encoding="utf-8"))["cells"]:
        if cell["cell_type"]=="code":
            result=get_ipython().run_cell("".join(cell["source"]))
            if result.error_in_exec:
                raise result.error_in_exec
import numpy as np
import pandas as pd
from IPython.display import display
ap_train=train.sub_type.eq("Daire")
ap_val=val.sub_type.eq("Daire")
size_train=pd.to_numeric(train["size"],errors="coerce")
size_val=pd.to_numeric(val["size"],errors="coerce")
known_train=ap_train & size_train.ge(15)
known_val=ap_val & size_val.ge(15)
missing_val=ap_val & ~size_val.ge(15)
print("Квартир train:",ap_train.sum(),"из них с известной площадью:",known_train.sum())
print("Квартир validation:",ap_val.sum(),"из них с известной площадью:",known_val.sum(),
      "без достоверной площади:",missing_val.sum())

Квартир train: 172033 из них с известной площадью: 110876
Квартир validation: 36593 из них с известной площадью: 24479 без достоверной площади: 12114


Известной считаем исходную площадь не меньше 15 м². Проверочные группы формируются до обучения моделей. Заполненная медианой площадь не считается известной.

## Обучение с одинаковым алгоритмом

В каждом варианте числовые пропуски заполняются медианой его train, категории кодируются One-Hot. Настройки LightGBM и логарифм целевой цены совпадают; меняются только строки обучения или использование площади.

In [2]:
import time
import warnings
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.metrics import mean_absolute_error,median_absolute_error,r2_score
from lightgbm import LGBMRegressor

def build_model(with_area=True):
    numbers=[c for c in num if with_area or c not in ("size_m2","size_missing")]
    encoder=ColumnTransformer([
        ("num",Pipeline([("fill",SimpleImputer(strategy="median")),
                         ("scale",StandardScaler())]),numbers),
        ("cat",Pipeline([("fill",SimpleImputer(strategy="most_frequent")),
                         ("onehot",OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                 min_frequency=50))]),cat)
    ],sparse_threshold=.8)
    reg=TransformedTargetRegressor(
        regressor=LGBMRegressor(random_state=42,verbosity=-1,n_jobs=4,num_leaves=15),
        func=np.log1p,inverse_func=np.expm1)
    return Pipeline([("encode",encoder),("model",reg)])

variants={
    "A: все, площадь заполнена":(ap_train,True),
    "B: только известная площадь":(known_train,True),
    "C: без площади":(ap_train,False),
}
models={}
for name,(mask,with_area) in variants.items():
    model=build_model(with_area)
    start=time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(X_train.loc[mask],train.loc[mask,"price"])
    models[name]=model
    print(name,"обучена на",mask.sum(),"строках за",round(time.perf_counter()-start,2),"с")

A: все, площадь заполнена обучена на 172033 строках за 2.98 с


B: только известная площадь обучена на 110876 строках за 1.92 с


C: без площади обучена на 172033 строках за 2.88 с


Во всех трёх вариантах цены validation не участвовали в обучении. Модель B видит меньше примеров, но каждый её тренировочный объект имеет настоящую площадь; A получает больше примеров, часть с приближённой площадью.

In [3]:
def evaluate(mask):
    y=val.loc[mask,"price"].to_numpy()
    rows=[]; preds={}
    for name,model in models.items():
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pred=model.predict(X_val.loc[mask])
        preds[name]=pred
        rows.append({"Модель":name,"Объектов":len(y),
                     "MAE, TRY":mean_absolute_error(y,pred),
                     "MedAE, TRY":median_absolute_error(y,pred),
                     "R²":r2_score(y,pred)})
    return pd.DataFrame(rows),preds,y
all_table,all_predictions,y_all=evaluate(ap_val)
known_table,known_predictions,y_known=evaluate(known_val)
missing_table,missing_predictions,y_missing=evaluate(missing_val)
print("Validation: все квартиры")
display(all_table.round(2))
print("Validation: квартиры с известной площадью")
display(known_table.round(2))
print("Validation: квартиры без достоверной площади")
display(missing_table.round(2))
print("R² без округления на всех квартирах:")
for _,row in all_table.iterrows():
    print(row["Модель"], round(row["R²"],6))


Validation: все квартиры


,Модель,Объектов,"MAE, TRY","MedAE, TRY",R²
0,"A: все, площадь заполнена",36593,169335.43,47229.37,0.0
1,B: только известная площадь,36593,171911.32,48379.69,0.0
2,C: без площади,36593,171143.06,48313.22,0.0


Validation: квартиры с известной площадью


,Модель,Объектов,"MAE, TRY","MedAE, TRY",R²
0,"A: все, площадь заполнена",24479,106120.77,46824.43,0.14
1,B: только известная площадь,24479,103820.87,45691.49,0.14
2,C: без площади,24479,108560.56,47640.70,0.13


Validation: квартиры без достоверной площади


,Модель,Объектов,"MAE, TRY","MedAE, TRY",R²
0,"A: все, площадь заполнена",12114,297074.54,48622.29,0.0
1,B: только известная площадь,12114,309503.04,55131.90,0.0
2,C: без площади,12114,297604.76,49977.31,0.0


R² без округления на всех квартирах:
A: все, площадь заполнена 0.001055
B: только известная площадь 0.001028
C: без площади 0.001008


На квартирах с известной площадью A и B сравниваются на одних объектах. Общая таблица показывает цену такого решения: модель, обученная без строк с пропуском, может проиграть на объектах с неизвестной площадью. Вариант C проверяет, полезен ли сам признак площади.


## Насколько устойчива разница

Для A и B на квартирах с известной площадью считаем парную разницу абсолютных ошибок. Повторно выбираем целые группы похожих объявлений, чтобы получить приближённый 95%-й интервал. Положительная разница означает преимущество B.

In [4]:
a_name="A: все, площадь заполнена"
b_name="B: только известная площадь"
c_name="C: без площади"
delta=np.abs(y_known-known_predictions[a_name])-np.abs(y_known-known_predictions[b_name])
group=pd.util.hash_pandas_object(
    val.loc[known_val,["address","sub_type","room_count","size"]],index=False).to_numpy()
aggregated=pd.DataFrame({"group":group,"delta":delta}).groupby("group").delta.agg(["sum","size"])
sums=aggregated["sum"].to_numpy(); counts=aggregated["size"].to_numpy()
rng=np.random.default_rng(42)
boot=[]
for _ in range(500):
    ids=rng.integers(0,len(aggregated),size=len(aggregated))
    boot.append(sums[ids].sum()/counts[ids].sum())
ci=np.quantile(boot,[.025,.975])
print("Преимущество B по MAE, TRY:",round(delta.mean(),2))
print("95%-й групповой интервал:",[float(round(x,2)) for x in ci])
print("Убедительное преимущество B:",bool(ci[0]>0))
print("Преимущество A над C на известных площадях:",
      round(known_table.loc[known_table.Модель.eq(c_name),"MAE, TRY"].iloc[0]
            -known_table.loc[known_table.Модель.eq(a_name),"MAE, TRY"].iloc[0],2),"TRY")

Преимущество B по MAE, TRY: 2299.9
95%-й групповой интервал: [1103.51, 3425.95]
Убедительное преимущество B: True
Преимущество A над C на известных площадях: 2439.78 TRY


## Интерпретация

На квартирах с известной площадью B снизила MAE относительно A примерно на 2 300 TRY; групповой 95%-й интервал преимущества B — от 1 104 до 3 426 TRY. Но на квартирах без площади A лучше B, а на всех квартирах A в целом точнее. Удалять все строки без площади при обучении нельзя считать общим решением. Площадь полезна там, где она известна; заполненная медианой площадь несёт меньше информации. Это сравнение стратегий, а не доказательство, что сам пропуск — единственная причина ошибок. Основную модель и test не меняем.

На всех квартирах validation R² почти нулевой (около 0,001) у всех трёх вариантов: здесь есть объявление о квартире за 2 млрд TRY, которое резко увеличивает дисперсию и квадрат ошибки. На квартирах с известной площадью R² около 0,14. Эти значения относятся к validation и не сопоставимы напрямую с R² старой общей модели на test.
